# 🧠 BERT Sentiment Analysis + Mixed Reviews + ABSA

### Complete Classroom Google Colab Notebook

This single notebook contains the complete journey:

1. Sentiment analysis with BERT
2. Tokenization and prediction
3. Confidence visualizations
4. Difficult examples: negation, sarcasm and mixed sentiment
5. Aspect-Based Sentiment Analysis (ABSA)
6. Mixed reviews such as **"I love chicken but hate KFC."**
7. Interactive classroom demos
8. Simple automatic aspect extraction
9. Final dashboard and project idea

**Run the notebook from top to bottom the first time.**

In [ ]:

# CELL 1 — Stable Google Colab installation
# ============================================================
# IMPORTANT:
# 1. Run this cell first.
# 2. This notebook does not need torchvision because it is text/NLP only.
# 3. We remove torchvision to avoid the common
#    "operator torchvision::nms does not exist" error.
# 4. Restart the runtime once after this cell.
# ============================================================

import sys
import subprocess

def run_pip(*packages):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", *packages
    ])

print("1/3 Removing unused torchvision...")
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision"],
    check=False
)

print("2/3 Installing stable NLP packages...")
run_pip(
    "transformers==4.46.3",
    "sentencepiece>=0.2.0",
    "spacy>=3.7,<4",
    "seaborn>=0.13",
    "matplotlib>=3.8",
    "pandas>=2.2,<3"
)

print("3/3 Installing spaCy English model...")
subprocess.check_call([
    sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"
])

print()
print("✅ Installation completed.")
print("🔄 IMPORTANT: Runtime → Restart session")
print("Then continue with Cell 2.")


1/3 Removing unused torchvision...
2/3 Installing stable NLP packages...


In [ ]:

# CELL 2 — Import and verify everything
# ============================================================

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
import torch
import transformers

from transformers import pipeline

print("✅ Environment loaded successfully!")
print()
print("Python       :", sys.version.split()[0])
print("PyTorch      :", torch.__version__)
print("Pandas       :", pd.__version__)
print("Transformers :", transformers.__version__)
print("spaCy        :", spacy.__version__)
print("GPU available:", torch.cuda.is_available())

# Load spaCy only after installation + restart.
nlp = spacy.load("en_core_web_sm")
print("spaCy model  : en_core_web_sm ✅")


In [ ]:

# CELL 3 — Test Transformers
# ============================================================

TEST_MODEL = "distilbert-base-uncased-finetuned-sst-2-english"
DEVICE = 0 if torch.cuda.is_available() else -1

test_pipeline = pipeline(
    "sentiment-analysis",
    model=TEST_MODEL,
    device=DEVICE
)

result = test_pipeline("I love teaching artificial intelligence!")

print("✅ Transformers pipeline is working!")
print("Device:", "GPU" if DEVICE == 0 else "CPU")
print(result)


## CELL 3 — What is Sentiment Analysis?

Sentiment analysis tries to determine the emotional orientation of text.

Typical classes:

- 🟢 POSITIVE
- 🔴 NEGATIVE

Examples:

**"The food was delicious." → POSITIVE**

**"The service was terrible." → NEGATIVE**

The interesting problem appears when a sentence contains more than one opinion.

In [ ]:

# CELL 4 — Load a pretrained BERT sentiment model

MODEL_NAME = "textattack/bert-base-uncased-SST-2"
DEVICE = 0 if torch.cuda.is_available() else -1

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    device=DEVICE
)

print("✅ BERT sentiment model loaded!")
print("Model:", MODEL_NAME)
print("Device:", "GPU" if DEVICE == 0 else "CPU")


## CELL 5 — BERT Pipeline Concept

Conceptually:

**Text → Tokenizer → Token IDs → BERT → Classifier → Sentiment**

The next drawing is a teaching diagram, not the exact internal computation graph.

In [ ]:
# CELL 6 — Draw the BERT sentiment pipeline

fig, ax = plt.subplots(figsize=(13, 3))
ax.axis("off")

boxes = [
    ("📝 Text", 0.05),
    ("🔤 Tokenizer", 0.27),
    ("🔢 Token IDs", 0.49),
    ("🧠 BERT", 0.68),
    ("🎯 Classifier", 0.86)
]

for label, x in boxes:
    ax.text(
        x, 0.5, label,
        ha="center", va="center", fontsize=15,
        bbox=dict(boxstyle="round,pad=0.5", facecolor="white", edgecolor="black")
    )

for x in [0.16, 0.38, 0.59, 0.77]:
    ax.annotate("→", xy=(x + 0.035, 0.5), xytext=(x - 0.01, 0.5),
                fontsize=25, ha="center", va="center")

plt.title("Conceptual BERT Sentiment Analysis Pipeline", fontsize=16)
plt.show()

In [ ]:
# CELL 7 — Tokenization demonstration

tokenizer = sentiment_pipeline.tokenizer
text = "I love this amazing chicken restaurant!"

tokens = tokenizer.tokenize(text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print("Original text:")
print(text)

print("\nTokens:")
print(tokens)

print("\nToken IDs:")
print(token_ids)

print("\nSpecial-token version:")
print(tokenizer(text))

## CELL 8 — Understanding the tokenizer

A tokenizer converts human language into units that the model can process.

BERT also uses special tokens such as:

- `[CLS]` — classification representation
- `[SEP]` — separates sequences
- `[PAD]` — padding when needed

The model ultimately processes numerical representations rather than raw English words.

In [ ]:
# CELL 9 — First BERT prediction

sentence = "I absolutely loved the food and the service."

result = sentiment_pipeline(sentence)[0]

print("Sentence:", sentence)
print("Predicted sentiment:", result["label"])
print("Confidence:", f"{result['score']:.2%}")

In [ ]:
# CELL 10 — Reusable prediction function

def explain_sentiment(text):
    prediction = sentiment_pipeline(text)[0]
    print("📝 Text:", text)
    print("🎯 Sentiment:", prediction["label"])
    print("📊 Confidence:", f"{prediction['score']:.2%}")
    return prediction

explain_sentiment("The movie was fantastic!")

In [ ]:
# CELL 11 — Classroom experiment

sentences = [
    "The food was delicious.",
    "The service was terrible.",
    "The movie was excellent.",
    "The product broke after one day.",
    "I really enjoyed the experience.",
    "I regret buying this product.",
    "The hotel room was clean and comfortable.",
    "The staff were rude and unhelpful."
]

rows = []
for text in sentences:
    prediction = sentiment_pipeline(text)[0]
    rows.append({
        "Text": text,
        "Sentiment": prediction["label"],
        "Confidence": prediction["score"]
    })

classroom_df = pd.DataFrame(rows)
classroom_df["Confidence %"] = (classroom_df["Confidence"] * 100).round(2)

display(classroom_df[["Text", "Sentiment", "Confidence %"]])

In [ ]:

# CELL 12 — Confidence visualization

plt.figure(figsize=(12, 6))

sns.barplot(
    data=classroom_df,
    x="Confidence",
    y="Text",
    hue="Sentiment",
    dodge=False
)

plt.xlim(0, 1)
plt.xlabel("Model confidence")
plt.ylabel("Sentence")
plt.title("BERT Sentiment Confidence")
plt.legend(title="Prediction")
plt.tight_layout()
plt.show()


In [ ]:
# CELL 13 — Positive vs Negative count

counts = classroom_df["Sentiment"].value_counts()

plt.figure(figsize=(7, 5))
counts.plot(kind="bar")
plt.xlabel("Sentiment")
plt.ylabel("Number of sentences")
plt.title("Sentiment Distribution")
plt.xticks(rotation=0)
plt.show()

## CELL 14 — Why sentiment can be difficult

Consider:

> **"The food was amazing, but the service was awful."**

There are two opinions:

- Food → positive
- Service → negative

A standard binary sentiment classifier may still produce only one overall label.

Other difficult examples:

- Negation: "The food was not good."
- Sarcasm: "Great! Another two-hour delay."
- Mixed sentiment: "I love the camera but hate the battery."
- Context: "It is small, but that is exactly what I wanted."

This motivates **Aspect-Based Sentiment Analysis (ABSA)**.

In [ ]:
# CELL 15 — Conceptual attention-style heatmap

# IMPORTANT:
# This is randomly generated for teaching.
# It is NOT the actual attention of the BERT model.

sentence = "I love the food but hate the service."
words = sentence.replace(".", "").split()

np.random.seed(7)
attention_demo = np.random.rand(len(words), len(words))

plt.figure(figsize=(10, 6))
sns.heatmap(
    attention_demo,
    xticklabels=words,
    yticklabels=words,
    cmap="viridis"
)

plt.title("Conceptual Attention-Style Heatmap")
plt.xlabel("Words")
plt.ylabel("Words")
plt.show()

print("⚠️ Educational visualization only — not actual BERT attention weights.")

In [ ]:
# CELL 16 — Simplified BERT architecture

print("\n".join([
    "BERT-style architecture",
    "        ↓",
    "Input Text",
    "        ↓",
    "Tokenizer",
    "        ↓",
    "Token + Position Representations",
    "        ↓",
    "Transformer Encoder",
    "   ├── Self-Attention",
    "   ├── Feed-Forward Network",
    "   ├── Residual Connections",
    "   └── Layer Normalization",
    "        ↓",
    "Contextual Representation",
    "        ↓",
    "Classification Layer",
    "        ↓",
    "POSITIVE / NEGATIVE"
]))

In [ ]:
# CELL 17 — Count model parameters

total_params = sum(p.numel() for p in sentiment_pipeline.model.parameters())
trainable_params = sum(
    p.numel() for p in sentiment_pipeline.model.parameters()
    if p.requires_grad
)

print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")

print("\n💡 Teaching point:")
print("A pretrained transformer contains many learned parameters.")
print("We can reuse those learned representations instead of training from scratch.")

In [ ]:
# CELL 18 — Tricky examples

tricky_sentences = [
    "I do not like this product.",
    "This is not bad.",
    "Great, another two-hour delay.",
    "I love the camera but hate the battery.",
    "The food was okay, nothing special.",
    "I expected better from such an expensive phone."
]

for text in tricky_sentences:
    prediction = sentiment_pipeline(text)[0]
    print(f"Text: {text}")
    print(f"Prediction: {prediction['label']} ({prediction['score']:.2%})")
    print("-" * 80)

# 🍗 CELL 19 — Mixed Reviews and the Need for ABSA

Consider:

> **"I love chicken but hate KFC."**

A single overall sentiment label is not enough to explain the sentence.

| Aspect | Sentiment |
|---|---|
| chicken | ❤️ Positive |
| KFC | 💔 Negative |

This motivates **Aspect-Based Sentiment Analysis (ABSA)**.

ABSA asks:

> **What sentiment is expressed toward a particular aspect?**

Conceptually:

**Sentence + Aspect → Aspect Sentiment**

In [ ]:

# CELL 20 — Load an ABSA model

# This model card documents text_pair usage for aspect-level sentiment.
ABSA_MODEL = "yangheng/deberta-v3-base-absa-v1.1"

absa_pipeline = pipeline(
    "text-classification",
    model=ABSA_MODEL,
    tokenizer=ABSA_MODEL,
    device=DEVICE
)

print("✅ ABSA model loaded!")
print("Model:", ABSA_MODEL)
print("Device:", "GPU" if DEVICE == 0 else "CPU")


In [ ]:
# CELL 21 — First ABSA example

sentence = "I love chicken but hate KFC."
aspects = ["chicken", "KFC"]

for aspect in aspects:
    result = absa_pipeline(sentence, text_pair=aspect)[0]
    print("Aspect:", aspect)
    print("Sentiment:", result["label"])
    print("Confidence:", f"{result['score']:.2%}")
    print("-" * 50)

## CELL 22 — Why provide an aspect?

Instead of asking:

> What is the sentiment of the whole sentence?

we ask:

> What is the sentiment with respect to **this aspect**?

For example:

`"I love chicken but hate KFC." + "chicken" → Positive`

`"I love chicken but hate KFC." + "KFC" → Negative`

In [ ]:
# CELL 23 — Reusable ABSA function

def analyze_aspects(sentence, aspects):
    results = []

    for aspect in aspects:
        prediction = absa_pipeline(
            sentence,
            text_pair=aspect
        )[0]

        results.append({
            "Aspect": aspect,
            "Sentiment": prediction["label"],
            "Confidence": prediction["score"]
        })

    return pd.DataFrame(results)

sentence = "I love chicken but hate KFC."
aspects = ["chicken", "KFC"]

result_df = analyze_aspects(sentence, aspects)
result_df["Confidence %"] = (result_df["Confidence"] * 100).round(2)

display(result_df[["Aspect", "Sentiment", "Confidence %"]])

In [ ]:
# CELL 24 — Complex restaurant review

sentence = (
    "The food was delicious, but the service was terrible "
    "and the restaurant was too expensive."
)

aspects = ["food", "service", "restaurant", "price"]

restaurant_df = analyze_aspects(sentence, aspects)
restaurant_df["Confidence %"] = (restaurant_df["Confidence"] * 100).round(2)

print("Review:")
print(sentence)
print()

display(restaurant_df[["Aspect", "Sentiment", "Confidence %"]])

In [ ]:

# CELL 25 — Aspect-level confidence visualization

plt.figure(figsize=(9, 5))

sns.barplot(
    data=restaurant_df,
    x="Aspect",
    y="Confidence",
    hue="Sentiment",
    dodge=False
)

plt.ylim(0, 1)
plt.ylabel("Confidence")
plt.title("ABSA Confidence by Aspect")
plt.tight_layout()
plt.show()


In [ ]:
# CELL 26 — Student prediction challenge

challenge_sentence = (
    "The camera quality is excellent, the battery is terrible, "
    "and the phone design is beautiful."
)

challenge_aspects = ["camera quality", "battery", "phone design"]

print("Read the review before running the model:")
print(challenge_sentence)
print()
print("Try to predict:")
print("camera quality → ?")
print("battery → ?")
print("phone design → ?")
print()

challenge_result = analyze_aspects(
    challenge_sentence,
    challenge_aspects
)

challenge_result["Confidence %"] = (
    challenge_result["Confidence"] * 100
).round(2)

display(challenge_result[["Aspect", "Sentiment", "Confidence %"]])

## CELL 27 — Interactive ABSA Classroom Demo

Type a review and provide aspects separated by commas.

Example:

Review:
`The laptop is fast but the battery is disappointing.`

Aspects:
`laptop, battery`

In [ ]:
# CELL 28 — Interactive ABSA

sentence = input("📝 Enter a review: ")

aspects_input = input("🔍 Enter aspects separated by commas: ")

aspects = [
    aspect.strip()
    for aspect in aspects_input.split(",")
    if aspect.strip()
]

if not aspects:
    print("⚠️ Please enter at least one aspect.")
else:
    interactive_df = analyze_aspects(sentence, aspects)
    interactive_df["Confidence %"] = (
        interactive_df["Confidence"] * 100
    ).round(2)

    display(interactive_df[["Aspect", "Sentiment", "Confidence %"]])

In [ ]:
# CELL 29 — Standard BERT vs ABSA

sentence = "I love chicken but hate KFC."

print("Sentence:")
print(sentence)

print("\n" + "=" * 70)
print("STANDARD BERT")
print("=" * 70)

overall = sentiment_pipeline(sentence)[0]

print("Overall sentiment:", overall["label"])
print("Confidence:", f"{overall['score']:.2%}")

print("\n" + "=" * 70)
print("ASPECT-BASED SENTIMENT ANALYSIS")
print("=" * 70)

comparison_df = analyze_aspects(sentence, ["chicken", "KFC"])
comparison_df["Confidence %"] = (
    comparison_df["Confidence"] * 100
).round(2)

display(comparison_df[["Aspect", "Sentiment", "Confidence %"]])

In [ ]:
# CELL 30 — Visual comparison: document-level vs aspect-level

fig, ax = plt.subplots(figsize=(13, 5))
ax.axis("off")

ax.text(0.25, 0.75, "STANDARD SENTIMENT",
        ha="center", fontsize=16, fontweight="bold")

ax.text(
    0.25, 0.55,
    '"I love chicken but hate KFC."',
    ha="center", fontsize=13,
    bbox=dict(boxstyle="round,pad=0.5", facecolor="white", edgecolor="black")
)

ax.text(0.25, 0.32, "ONE OVERALL LABEL",
        ha="center", fontsize=14)

ax.text(0.75, 0.75, "ASPECT-BASED SENTIMENT",
        ha="center", fontsize=16, fontweight="bold")

ax.text(
    0.75, 0.58,
    "chicken → Positive\nKFC → Negative",
    ha="center", fontsize=13,
    bbox=dict(boxstyle="round,pad=0.5", facecolor="white", edgecolor="black")
)

ax.text(0.75, 0.32, "MULTIPLE ASPECT LABELS",
        ha="center", fontsize=14)

ax.annotate("", xy=(0.57, 0.55), xytext=(0.43, 0.55),
            arrowprops=dict(arrowstyle="->", lw=2))

plt.title("Document-Level Sentiment vs Aspect-Level Sentiment", fontsize=17)
plt.show()

# 🔎 CELL 31 — Simple Automatic Aspect Extraction

So far, we manually supplied aspects.

For teaching purposes, spaCy can find **nouns and proper nouns** as candidate aspects.

Examples:

- food
- service
- battery
- camera
- restaurant

⚠️ This is a simple educational heuristic, not a production-quality aspect extractor.

In [ ]:

# CELL 32 — Verify the spaCy English model

print("spaCy pipeline:", nlp.pipe_names)
print("✅ spaCy English model is ready.")


In [ ]:
# CELL 33 — Extract candidate aspects

sentence = (
    "The camera is excellent, the battery is terrible, "
    "and the phone design is beautiful."
)

doc = nlp(sentence)

possible_aspects = []

for token in doc:
    if token.pos_ in ["NOUN", "PROPN"]:
        if token.text.lower() not in [
            aspect.lower() for aspect in possible_aspects
        ]:
            possible_aspects.append(token.text)

print("Sentence:")
print(sentence)

print("\nCandidate aspects:")
print(possible_aspects)

In [ ]:
# CELL 34 — Visualize aspect extraction

doc = nlp(sentence)

words = [token.text for token in doc]
pos_tags = [token.pos_ for token in doc]

fig, ax = plt.subplots(figsize=(14, 5))
ax.axis("off")

for i, (word, pos) in enumerate(zip(words, pos_tags)):
    is_candidate = pos in ["NOUN", "PROPN"]

    ax.text(
        i, 0.55, word,
        ha="center", fontsize=14,
        bbox=dict(
            boxstyle="round,pad=0.35",
            facecolor="white" if not is_candidate else "lightgreen",
            edgecolor="black"
        )
    )

    ax.text(i, 0.25, pos, ha="center", fontsize=11)

ax.set_xlim(-1, len(words))
ax.set_ylim(0, 1)
ax.set_title("Simple POS-Based Candidate Aspect Extraction", fontsize=16)
plt.show()

print("🟩 Green boxes = noun/proper-noun candidates")

In [ ]:

# CELL 35 — Automatic ABSA

def automatic_absa(sentence):
    """
    Educational automatic ABSA:
    1. Extract noun/proper-noun candidates with spaCy.
    2. Send each candidate as text_pair to the ABSA classifier.

    This is a classroom prototype, not a production-grade aspect extractor.
    """
    doc = nlp(sentence)

    aspects = []
    seen = set()

    for token in doc:
        if token.pos_ in ["NOUN", "PROPN"]:
            key = token.text.lower()
            if key not in seen:
                aspects.append(token.text)
                seen.add(key)

    results = []

    for aspect in aspects:
        prediction = absa_pipeline(
            sentence,
            text_pair=aspect
        )[0]

        results.append({
            "Aspect": aspect,
            "Sentiment": prediction["label"],
            "Confidence": prediction["score"]
        })

    return pd.DataFrame(
        results,
        columns=["Aspect", "Sentiment", "Confidence"]
    )

auto_result = automatic_absa(sentence)

if not auto_result.empty:
    auto_result["Confidence %"] = (
        auto_result["Confidence"] * 100
    ).round(2)
    display(auto_result[["Aspect", "Sentiment", "Confidence %"]])
else:
    print("No noun/proper-noun aspect candidates were found.")


In [ ]:
# CELL 36 — Realistic mixed product review

product_review = (
    "The laptop has an excellent display and a fast processor, "
    "but the battery life is disappointing and the keyboard feels cheap."
)

print("Review:")
print(product_review)

product_result = automatic_absa(product_review)

if len(product_result) > 0:
    product_result["Confidence %"] = (
        product_result["Confidence"] * 100
    ).round(2)

display(product_result[["Aspect", "Sentiment", "Confidence %"]])

In [ ]:
# CELL 37 — Manual aspects can be more precise

manual_aspects = [
    "display",
    "processor",
    "battery life",
    "keyboard"
]

manual_result = analyze_aspects(
    product_review,
    manual_aspects
)

manual_result["Confidence %"] = (
    manual_result["Confidence"] * 100
).round(2)

display(manual_result[["Aspect", "Sentiment", "Confidence %"]])

## CELL 38 — Complete ABSA pipeline

A practical ABSA system can be understood as:

**Review**
↓
**Aspect Extraction**
↓
**Aspect Identification**
↓
**Aspect Sentiment Classification**
↓
**Structured Output / Dashboard**

Example:

```text
"The camera is excellent but the battery is terrible."

camera  → positive
battery → negative
```

The aspect-extraction and sentiment-classification steps can be separate components.

In [ ]:
# CELL 39 — Final ABSA dashboard

final_review = (
    "The food was delicious, the staff were friendly, "
    "but the service was slow and the prices were too high."
)

final_aspects = ["food", "staff", "service", "prices"]

final_result = analyze_aspects(
    final_review,
    final_aspects
)

final_result["Confidence %"] = (
    final_result["Confidence"] * 100
).round(2)

print("📝 FINAL REVIEW")
print(final_review)

display(final_result[["Aspect", "Sentiment", "Confidence %"]])

In [ ]:

# CELL 40 — Final dashboard visualization

plt.figure(figsize=(10, 6))

sns.barplot(
    data=final_result,
    x="Aspect",
    y="Confidence",
    hue="Sentiment",
    dodge=False
)

plt.ylim(0, 1)
plt.xlabel("Aspect")
plt.ylabel("Confidence")
plt.title("Final ABSA Dashboard")
plt.tight_layout()
plt.show()


In [ ]:
# CELL 41 — End-to-end workflow drawing

fig, ax = plt.subplots(figsize=(14, 4))
ax.axis("off")

steps = [
    ("📝 Review", 0.08),
    ("🔍 Find\naspects", 0.28),
    ("🧠 Transformer", 0.50),
    ("🎯 Sentiment\nper aspect", 0.72),
    ("📊 Dashboard", 0.92)
]

for label, x in steps:
    ax.text(
        x, 0.5, label,
        ha="center", va="center", fontsize=14,
        bbox=dict(boxstyle="round,pad=0.5", facecolor="white", edgecolor="black")
    )

for x in [0.18, 0.38, 0.60, 0.82]:
    ax.annotate("→", xy=(x + 0.035, 0.5), xytext=(x - 0.01, 0.5),
                fontsize=24, ha="center")

plt.title("End-to-End Aspect-Based Sentiment Analysis Workflow", fontsize=17)
plt.show()

# 🎓 CELL 42 — Mini Classroom Assessment

### Conceptual questions

1. What is sentiment analysis?
2. What is BERT?
3. What does a tokenizer do?
4. What is a pretrained model?
5. What is confidence?
6. Why can a mixed review be difficult for binary sentiment classification?
7. What is ABSA?
8. Why do we provide an aspect to an ABSA model?
9. What is the difference between document-level and aspect-level sentiment?
10. Why is simple noun extraction not a complete aspect-extraction solution?

### Coding challenge

Analyze:

> **"The camera takes amazing photos, but the battery drains quickly and the screen is too dim."**

Try these aspects:

- camera
- photos
- battery
- screen

### Project idea

Build a **Product Review Intelligence System** that:

- accepts customer reviews
- extracts candidate aspects
- predicts aspect-level sentiment
- calculates confidence
- visualizes positive and negative aspects
- produces a dashboard

# 🏁 CELL 43 — Final Summary

### Standard BERT sentiment analysis

**Text → Tokenizer → BERT → Classification → Overall sentiment**

### Mixed reviews

One review may contain several opinions.

Example:

> "I love chicken but hate KFC."

### ABSA

**Sentence + Aspect → Aspect Sentiment**

```text
chicken → positive
KFC     → negative
```

### Big takeaway

**Ordinary sentiment analysis asks:**

> What is the overall sentiment?

**ABSA asks:**

> What is the sentiment toward each aspect?

This distinction is a useful foundation for practical NLP applications.


# ⚠️ Instructor Notes

### Recommended execution order

1. Run **Cell 1**.
2. When installation finishes, use **Runtime → Restart session**.
3. Start again from **Cell 2** and run downward.

### Why is a runtime restart required?

Package installation changes the Python environment. Restarting makes Colab reload
the newly installed versions cleanly.

### Important technical choice

This notebook deliberately does **not** install or import `torchvision`.
The notebook performs text/NLP tasks only. Avoiding the unused computer-vision package
prevents the common PyTorch/torchvision `operator torchvision::nms does not exist`
compatibility problem.

### Models used

- Document-level sentiment: `textattack/bert-base-uncased-SST-2`
- Aspect-based sentiment: `yangheng/deberta-v3-base-absa-v1.1`

The ABSA model is used with `text_pair=aspect`, as documented by its Hugging Face
model card.

### Classroom note

The spaCy noun/proper-noun method in the automatic ABSA section is intentionally simple.
It is useful for teaching the pipeline:

**Review → Candidate Aspects → Aspect Sentiment → Dashboard**

It should not be presented as a production-quality aspect extraction system.

### Performance note

The first execution downloads model files from Hugging Face. Later executions in
the same Colab environment can reuse the cached files.

### Interactive cell

The interactive ABSA cell uses `input()`. If you use **Run all**, Colab will pause
there until the student enters a review and its aspects. That is expected.
